In [10]:
from abc import ABC
import sys, os
import argparse

import xgboost as xgb
import pandas as pd
from xgbt_train import build_X


In [ ]:
# # モジュールの相対参照制限を強制的に回避
# current_dir = os.path.dirname(os.path.abspath(__file__))
# sys.path.append(os.path.join(current_dir, '..', 'analysis'))
# from xgbt_train import build_X

NameError: name '__file__' is not defined

In [11]:
TARGET = "stroke_flag"
NUM_FEATURES = 21
NUM_CLASSES = 2

In [12]:
class Attack_Di_Base(ABC):
    def __init__(self, path_to_xgbt_model_json):
        """
        攻撃者の初期化

        path_to_xgbt_model_json: 学習済みのxgboostモデルのjsonファイルへのパス
        """
        # json fileを読み込み
        xgbt_model = xgb.Booster()
        xgbt_model.load_model(path_to_xgbt_model_json)

        self.xgbt_model = xgbt_model

        self.X = None
        self.y = None
        self.inferred = None
    
    def infer(self, path_to_Ai_csv):
        Ai_df = pd.read_csv(path_to_Ai_csv, dtype=str, 
                            keep_default_na=False)
        
        # 説明変数と目的変数に分割
        X = build_X(Ai_df, TARGET)
        X.columns = self.xgbt_model.feature_names
        self.X = X.copy()
        self.y = pd.to_numeric(Ai_df[TARGET], errors="coerce").astype(int).values

        # print(set(self.xgbt_model.feature_names)-set(X.columns.tolist()))

        return None
    
    def save_inferred(self, path_to_output):
        if self.inferred is None:
            print("inferred is None. No file was saved.")
        else:
            self.inferred.to_csv(path_to_output, index=False, header=False)
            print("inferred was successfully saved.")

In [13]:
class Pred_Attack(Attack_Di_Base):
    """
    モデルが正答した行をmemberと推定する
    """
    def __init__(self, path_to_xgboost_model_json):
        super().__init__(path_to_xgboost_model_json)

    def infer(self, path_to_Ai_csv):
        super().infer(path_to_Ai_csv)

        pred = self.xgbt_model.predict(xgb.DMatrix(self.X))
        pred[pred<0.5] = 0
        pred[pred>=0.5] = 1
        
        inferred = pd.DataFrame(pred == self.y, dtype=int)
        self.inferred = inferred

        return inferred

In [14]:
class Conf_Attack(Attack_Di_Base):
    """
    モデルが確信を持って正答した行をmemberと推定する
    """
    def __init__(self, path_to_xgboost_model_json, threshold=0.1):
        super().__init__(path_to_xgboost_model_json)
        self.threshold = threshold

    def infer(self, path_to_Ai_csv):
        super().infer(path_to_Ai_csv)

        pred = self.xgbt_model.predict(xgb.DMatrix(self.X))
        confidence = pd.DataFrame(pred-self.y).abs()
        inferred = (confidence <= self.threshold)
        self.inferred = inferred

        return inferred

In [ ]:
# if __name__ == "__main__":
#     ap = argparse.ArgumentParser(description="")
#     ap.add_argument("model_json", help="trained model JSON (Booster.save_model)")
#     ap.add_argument("Ai_csv", help="Ai.csv to attack")
#     args = ap.parse_args()

#     attacker = Pred_Attack(args.model_json)
#     pred = attacker.infer(args.Ai_csv)
#     attacker.save_inferred("inferred_membership1.csv")

#     attacker = Conf_Attack(args.model_json)
#     pred = attacker.infer(args.Ai_csv)
#     attacker.save_inferred("inferred_membership2.csv")

In [20]:
id = "01"
model_json = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\D{id}.json"
Ai_csv = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\A{id}.csv"
out = "C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out"
out_pred = f"{out}\\inferred_membership1.csv"
out_conf = f"{out}\\inferred_membership2.csv"

In [19]:
attacker = Pred_Attack(model_json)
pred = attacker.infer(Ai_csv)
attacker.save_inferred(out_pred)

attacker = Conf_Attack(model_json)
pred = attacker.infer(Ai_csv)
attacker.save_inferred(out_conf)

inferred was successfully saved.
inferred was successfully saved.


In [23]:
import pandas as pd
filename = logfile_pre_attack_Di = "C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\log_attack_Di.txt"
for id_int in range(1, 21):
    print(id_int)
    id = f"{id_int:02d}"
    model_json = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\D{id}.json"
    Ai_csv = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\A{id}.csv"
    out = "C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out"
    out_pred = f"{out}\\inferred_membership1.csv"
    out_conf = f"{out}\\inferred_membership2.csv"
    attacker = Pred_Attack(model_json)
    pred = attacker.infer(Ai_csv)
    attacker.save_inferred(out_pred)
    count_pred = pd.read_csv(out_pred, header=None).value_counts()
    with open(filename, 'a') as f:
        print(f"{id}: {count_pred}", file=f)
    attacker = Conf_Attack(model_json)
    pred = attacker.infer(Ai_csv)
    attacker.save_inferred(out_conf)
    count_conf = pd.read_csv(out_conf, header=None).value_counts()
    with open(filename, 'a') as f:
        print(f"{id}: {count_conf}", file=f)

id_int = 22
id = f"{id_int:02d}"
model_json = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\D{id}.json"
Ai_csv = f"C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out\\PWSCUP2025_Pre_Data_for_Attack\\A{id}.csv"
out = "C:\\Users\\kikus\\Documents\\統数研\\pwscup2025-scripts\\out"
out_pred = f"{out}\\inferred_membership1.csv"
out_conf = f"{out}\\inferred_membership2.csv"
attacker = Pred_Attack(model_json)
pred = attacker.infer(Ai_csv)
attacker.save_inferred(out_pred)
count_pred = pd.read_csv(out_pred, header=None).value_counts()
with open(filename, 'a') as f:
    print(f"{id}: {count_pred}", file=f)
attacker = Conf_Attack(model_json)
pred = attacker.infer(Ai_csv)
attacker.save_inferred(out_conf)
count_conf = pd.read_csv(out_conf, header=None).value_counts()
with open(filename, 'a') as f:
    print(f"{id}: {count_conf}", file=f)

1
inferred was successfully saved.
inferred was successfully saved.
2
inferred was successfully saved.
inferred was successfully saved.
3
inferred was successfully saved.
inferred was successfully saved.
4
inferred was successfully saved.
inferred was successfully saved.
5
inferred was successfully saved.
inferred was successfully saved.
6
inferred was successfully saved.
inferred was successfully saved.
7
inferred was successfully saved.
inferred was successfully saved.
8
inferred was successfully saved.
inferred was successfully saved.
9
inferred was successfully saved.
inferred was successfully saved.
10
inferred was successfully saved.
inferred was successfully saved.
11
inferred was successfully saved.
inferred was successfully saved.
12
inferred was successfully saved.
inferred was successfully saved.
13
inferred was successfully saved.
inferred was successfully saved.
14
inferred was successfully saved.
inferred was successfully saved.
15


ValueError: Length mismatch: Expected axis has 21 elements, new values have 27 elements